[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/fall-2026/notebooks/NB04_pairwise_alignment.ipynb)

> **Opening this notebook from Canvas:** Select **File → Save a copy in Drive** before you begin. Work in your saved copy—not in the repository preview.

**Notebook ID:** `NB04_pairwise_alignment`  
**Course release:** Fall 2026  
**Version:** 1.1  
**Updated:** 2026-09-04


# NB04 - First pairwise alignments with Biopython

## Cytochrome c: sequence comparison now, structure comparison later

**Biological question:** What changes when we ask for the best alignment across two complete proteins versus the best matching region within them?

**Inputs**

- `Data/NB04_pairwise_alignment/cytochrome_c_course_set.fasta`

**Outputs**

- global and local alignment text files
- `motif_summary.tsv`
- `run_parameters.tsv`
- `results_summary.txt`

> Run the notebook from top to bottom. The setup automatically recognizes the standard student and instructor course-folder locations. Today, focus on the inputs, outputs, and biological decision—not on memorizing Python syntax.


## 1. Add and import the tools

Python is already running in Colab. `%pip install` adds Biopython, and `import` makes its sequence and alignment tools available.


In [ ]:
%pip install -q biopython

from pathlib import Path
from urllib.request import urlretrieve
import csv
import re

from Bio import Align, SeqIO

print("Biopython is ready.")


## 2. Connect Colab to Google Drive

Authorize the Google account that contains your course folder.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

print("Google Drive is connected.")


## 3. Locate this notebook's input and output folders

The setup recognizes both supported course-folder locations automatically:

- students: `MyDrive/BIOINFO4-5203-F26/`
- instructor: `MyDrive/Teaching/BIOINFO4-5203-F26/`

If neither exists, it creates the standard student location. Do not change individual data or output paths.

In [ ]:
# ===== COURSE SETTINGS: normally no editing is required =====
COURSE_FOLDER_NAME = "BIOINFO4-5203-F26"
COURSE_RELEASE = "fall-2026"
NOTEBOOK_ID = "NB04_pairwise_alignment"
NOTEBOOK_VERSION = "1.1"

# Students normally keep the course folder directly in MyDrive.
# The second location supports the instructor's existing Teaching folder.
candidate_course_dirs = [
    Path("/content/drive/MyDrive") / COURSE_FOLDER_NAME,
    Path("/content/drive/MyDrive/Teaching") / COURSE_FOLDER_NAME,
]
existing_course_dirs = [path for path in candidate_course_dirs if path.exists()]

if existing_course_dirs:
    COURSE_DIR = existing_course_dirs[0]
else:
    COURSE_DIR = candidate_course_dirs[0]
    COURSE_DIR.mkdir(parents=True, exist_ok=True)
    print("Created a new standard course folder.")

DATA_DIR = COURSE_DIR / "Data" / NOTEBOOK_ID
OUTPUT_DIR = COURSE_DIR / "Outputs" / NOTEBOOK_ID
FASTA_PATH = DATA_DIR / "cytochrome_c_course_set.fasta"

DATA_URL = (
    "https://raw.githubusercontent.com/RobBurnap/"
    "Bioinformatics-MICR4203-MICR5203/"
    f"{COURSE_RELEASE}/data/{NOTEBOOK_ID}/cytochrome_c_course_set.fasta"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Course folder :", COURSE_DIR)
print("Data folder   :", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("FASTA input   :", FASTA_PATH)


## 4. Obtain and read the FASTA file

The notebook first looks for the class FASTA file in your Drive. If it is missing, it downloads the public course copy from GitHub into the correct `Data/NB04_pairwise_alignment/` folder. It never overwrites an existing copy.

It then verifies the expected record count and reports every sequence identifier and length before analysis.

In [ ]:
if FASTA_PATH.exists():
    print("Using the existing FASTA file in your Drive.")
else:
    print("The FASTA file is missing; downloading the course copy from GitHub...")
    try:
        urlretrieve(DATA_URL, FASTA_PATH)
    except Exception as error:
        raise RuntimeError(
            "The course FASTA file could not be downloaded. "
            "Check your internet connection and ask the instructor before changing paths."
        ) from error
    print("Downloaded:", FASTA_PATH.name)

records = list(SeqIO.parse(FASTA_PATH, "fasta"))
assert len(records) == 3, f"Expected 3 FASTA records; found {len(records)}."

for record in records:
    print(record.id, "->", len(record.seq), "amino acids")


## 5. Select two sequences

For the first comparison, we will align cyanobacterial cytochrome c549 with mature human cytochrome c. Their structures share a cytochrome c fold, but their sequences differ substantially in length and composition.

The human UniProt sequence begins with an initiator methionine. `[1:]` selects the mature sequence beginning with the following glycine, matching the chain used in the later PyMOL comparison.


In [ ]:
record_by_id = {record.id: record for record in records}

sequence_1 = record_by_id["1E29_A_Cytochrome_c549_Synechocystis|PDB:1E29|chain:A"].seq
sequence_2 = record_by_id["P99999_Human_cytochrome_c|UniProt:P99999|organism:Homo_sapiens"].seq[1:]

print("Sequence 1 length:", len(sequence_1))
print("Sequence 2 length:", len(sequence_2))


## 6. Global alignment: Needleman-Wunsch

A global alignment asks the algorithm to account for both sequences from beginning to end.

Our introductory scoring system is intentionally simple:

- identical amino acids: `+2`
- different amino acids: `-1`
- each gap position: `-2`


In [ ]:
aligner = Align.PairwiseAligner()
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.gap_score = -2
aligner.mode = "global"

global_alignment = aligner.align(sequence_1, sequence_2)[0]

print("Mode:", aligner.mode)
print("Algorithm:", aligner.algorithm)
print("Score:", global_alignment.score)
print(global_alignment)


### Observe before continuing

- Does the alignment cover both complete sequences?
- Where were gaps introduced?
- Are matching positions evenly distributed?
- Why can a global score be negative for these divergent, unequal-length proteins?

`|` marks identity, `.` marks a mismatch, and `-` marks a gap. Alignment scores are meaningful only with the scoring system and alignment mode reported alongside them.

## 7. Local alignment: Smith-Waterman

Now change only the mode. A local alignment searches for the strongest matching region within the sequences.


In [ ]:
aligner.mode = "local"

local_alignment = aligner.align(sequence_1, sequence_2)[0]

print("Mode:", aligner.mode)
print("Algorithm:", aligner.algorithm)
print("Score:", local_alignment.score)
print(local_alignment)


## 8. Interpret the difference

1. Which regions disappeared from the local alignment?

   **Your answer:**

2. Which alignment would you choose for proteins expected to be homologous across their complete lengths? Why?

   **Your answer:**

3. Which alignment would you choose to find one conserved region inside a longer protein? Why?

   **Your answer:**

4. Why is choosing whichever alignment has the larger score not a sufficient biological decision rule?

   **Your answer:**


## 9. Compare a closer pair: tuna and human cytochrome c

The PDB FASTA for 3CYT begins with `X`, representing its modified/nonstandard N-terminal position. We omit that position for this comparison. Predict whether a global alignment will now be more biologically natural.


In [ ]:
tuna_mature = record_by_id["3CYT_Tuna_cytochrome_c|PDB:3CYT|organism:Thunnus_alalunga"].seq[1:]
human_mature = record_by_id["P99999_Human_cytochrome_c|UniProt:P99999|organism:Homo_sapiens"].seq[1:]

aligner.mode = "global"
tuna_human_alignment = aligner.align(tuna_mature, human_mature)[0]

print("Algorithm:", aligner.algorithm)
print("Score:", tuna_human_alignment.score)
print(tuna_human_alignment)


## 10. Preview the sequence-to-structure connection

All three proteins contain a `CXXCH` heme-binding motif. In the regular expression `C..CH`, each dot means “any amino acid.” The motif will be revisited in the later PyMOL structural comparison.


In [ ]:
motif_rows = []

for record in records:
    motif = re.search(r"C..CH", str(record.seq))
    if motif:
        row = [record.id, motif.group(), motif.start() + 1, motif.end()]
        motif_rows.append(row)
        print(record.id, "->", motif.group(), "at positions", motif.start() + 1, "to", motif.end())
    else:
        print(record.id, "-> motif not found")


## 11. Save reproducible outputs

The notebook writes results to `Outputs/NB04_pairwise_alignment/`. It does not modify the FASTA file in `Data/`.


In [ ]:
(OUTPUT_DIR / "global_1E29_vs_human.txt").write_text(
    str(global_alignment) + f"\nScore: {global_alignment.score}\n"
)

(OUTPUT_DIR / "local_1E29_vs_human.txt").write_text(
    str(local_alignment) + f"\nScore: {local_alignment.score}\n"
)

(OUTPUT_DIR / "global_tuna_vs_human.txt").write_text(
    str(tuna_human_alignment) + f"\nScore: {tuna_human_alignment.score}\n"
)

with open(OUTPUT_DIR / "motif_summary.tsv", "w", newline="") as handle:
    writer = csv.writer(handle, delimiter="\t")
    writer.writerow(["sequence_id", "motif", "start_1_based", "end_1_based"])
    writer.writerows(motif_rows)

with open(OUTPUT_DIR / "run_parameters.tsv", "w", newline="") as handle:
    writer = csv.writer(handle, delimiter="\t")
    writer.writerow(["parameter", "value"])
    writer.writerows([
        ["notebook_id", NOTEBOOK_ID],
        ["notebook_version", NOTEBOOK_VERSION],
        ["course_release", COURSE_RELEASE],
        ["notebook_version", NOTEBOOK_VERSION],
        ["course_release", COURSE_RELEASE],
        ["input_fasta", FASTA_PATH.name],
        ["match_score", 2],
        ["mismatch_score", -1],
        ["gap_score", -2],
        ["global_algorithm", "Needleman-Wunsch"],
        ["local_algorithm", "Smith-Waterman"],
    ])

(OUTPUT_DIR / "results_summary.txt").write_text(
    "NB04 PAIRWISE ALIGNMENT SUMMARY\n"
    f"Input records: {len(records)}\n"
    f"Global 1E29-human score: {global_alignment.score}\n"
    f"Local 1E29-human score: {local_alignment.score}\n"
    f"Global tuna-human score: {tuna_human_alignment.score}\n"
    f"CXXCH motifs found: {len(motif_rows)}\n"
)

print("Files created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Exit ticket

Complete this sentence:

> I would choose a __________ alignment when __________ because __________.

**Later connection:** these same sequences will be compared structurally in PyMOL to ask whether a shared fold can persist despite weaker sequence similarity.
